# DACON 236749 — existing-only 9504 rows, H100 end-to-end (Run All)

**What Run All does** (every stage is resume-safe; re-running Run All continues where it stopped):

1. **preflight** — H100 via `nvidia-smi`, object volume `/shared` and `deployment.json`, no other GPU process, disk budget, package-index reachability.
2. **stage_inputs** — verifies every immutable deployment file (bytes + SHA-256), the runtime manifest SHA, the 9504-row / 6652·1426·1426 / per-split FILE_FAKE balance / exposure / singing-train-only / provenance+content leakage contract; extracts **only the selected audio members** from hash-verified shards and verifies every file; extracts the 27 frozen assets and the original `source/` modules (read-only).
3. **environment** — isolated, hash-pinned Python 3.11 venv (uv 0.8.17; torch/torchaudio 2.9.1, torchcodec 0.8.1, transformers 4.57.6, numpy 2.2.6, … as in the reference requirements; **no fairseq**), exact freeze check, in-venv probe of the H100 and of the torchaudio→torchcodec decoder (FFmpeg libraries provisioned only if missing).
4. **pilot** — H100 VRAM-measured MERT batch (80 % VRAM, ≥90 % throughput rule) and a representative train-only runtime pilot with a projection for all 9504 rows against the ~16 h target (reported, never enforced; no data is dropped).
5. **features** — resume-safe full-duration HTDemucs → MERT-330M layer-12 windows, DF-Arena fc5-input embeddings, PANNs presence for all 9504 rows (exact coverage check).
6. **train** — unchanged WindowMIL (music) and DF-Arena `fc5` (voice) heads, seeds 41/42/43, train-only normalisation, selection on **val only**, then an immutable `candidate_lock.json` (six checkpoint hashes). Test features are never opened before the lock.
7. **test_eval** — verifies the lock, scores test once with the locked heads; masked known-label metrics.
8. **parity** — fresh-process raw-audio re-extraction of representative samples, compared with the cache and the val/test predictions.
9. **handoff** — the single `DACON_Existing9504_H100_Handoff.zip` (checkpoints, predictions, metrics, manifest/provenance/audit, code, notebook, environment lock, assets, measurements, logs, parity samples, SHA256SUMS), CRC+SHA verified, bound to the lock/test/parity evidence, atomically published on the object volume, and exposed as a hash-verified download next to this notebook.

On any exception a `DACON_Existing9504_H100_Handoff_FAILED.zip` with status `failed` is written and the error is re-raised.
This notebook does **not** start/stop services, use API keys, or submit anything to DACON. No cell needs editing.

Caveats carried from the dataset README: val/test are *not* independent (historical exposure 997/1426 and 1001/1426); singing is train-only; music source/length bias remains; no official/L4 score is claimed.


## 0 · Locate and verify the author code

In [ ]:
import hashlib, json, os, shutil, stat, sys, zipfile
from pathlib import Path, PurePosixPath

AUTHOR_CODE_SHA256 = "5994d3f2e442c5721eff5e8844b0ca72a61a7ad1253094eb8066241b27d2364e"
NOTEBOOK_NAME = "DACON_Existing9504_H100.ipynb"
DEPLOYMENT_ROOT = Path(os.environ.get("DACON_DEPLOYMENT_ROOT", "/shared/existing-only-9504-h100-20260920"))
if not (DEPLOYMENT_ROOT / "deployment.json").is_file() and (Path.cwd() / "deployment.json").is_file():
    DEPLOYMENT_ROOT = Path.cwd()  # the notebook lives in the deployment root
JOB_ROOT = Path(os.environ.get("DACON_JOB_ROOT", "/root/dacon_existing9504_h100_20260920"))
_session = os.environ.get("JPY_SESSION_NAME", "")
NOTEBOOK_PATH = Path(_session) if _session.endswith(".ipynb") and Path(_session).is_file() else DEPLOYMENT_ROOT / NOTEBOOK_NAME


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _extract_author_code(archive_path: Path, destination: Path) -> None:
    """Traversal/symlink/duplicate-safe extraction into a fresh directory, then atomic rename."""
    staging = destination.with_name(destination.name + ".partial")
    if staging.exists():
        shutil.rmtree(staging)
    staging.mkdir(parents=True)
    with zipfile.ZipFile(archive_path) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)):
            raise RuntimeError("author_code.zip has duplicate members")
        for info in archive.infolist():
            relative = PurePosixPath(info.filename)
            if (relative.is_absolute() or ".." in relative.parts or "\\" in info.filename
                    or stat.S_ISLNK(info.external_attr >> 16)):
                raise RuntimeError(f"Unsafe author_code.zip member: {info.filename}")
            target = (staging / Path(*relative.parts)).resolve()
            if not target.is_relative_to(staging.resolve()):
                raise RuntimeError(f"author_code.zip member escapes: {info.filename}")
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as source, target.open("wb") as sink:
                shutil.copyfileobj(source, sink)
    (staging / ".extracted").write_text(AUTHOR_CODE_SHA256 + "\n")
    if destination.exists():
        shutil.rmtree(destination)
    staging.rename(destination)


CODE_ZIP = DEPLOYMENT_ROOT / "author_code.zip"
if not CODE_ZIP.is_file():
    raise FileNotFoundError(f"author_code.zip must sit next to this notebook in {DEPLOYMENT_ROOT}")
_actual = _sha256(CODE_ZIP)
if _actual != AUTHOR_CODE_SHA256:
    raise RuntimeError(f"author_code.zip SHA-256 {_actual} != notebook-bound {AUTHOR_CODE_SHA256}")
CODE_DIR = JOB_ROOT / "code" / AUTHOR_CODE_SHA256[:16]
_marker = CODE_DIR / ".extracted"
if not _marker.is_file() or _marker.read_text().strip() != AUTHOR_CODE_SHA256:
    _extract_author_code(CODE_ZIP, CODE_DIR)
for _name in [n for n in sys.modules if n == "dacon9504a100" or n.startswith("dacon9504a100.")]:
    del sys.modules[_name]
sys.path.insert(0, str(CODE_DIR))
from dacon9504a100.orchestrator import Pipeline, build_config

_config = build_config(DEPLOYMENT_ROOT, JOB_ROOT, CODE_DIR, CODE_ZIP, NOTEBOOK_PATH)
_config["workspace"] = json.loads((DEPLOYMENT_ROOT / "deployment.json").read_text())["workspace"]
pipe = Pipeline(config=_config)
print(f"deployment root : {DEPLOYMENT_ROOT}")
print(f"job root        : {JOB_ROOT}")
print(f"author code     : {CODE_DIR} (sha256 {AUTHOR_CODE_SHA256})")
print(f"notebook        : {NOTEBOOK_PATH}")
print(f"kernel python   : {sys.version.split()[0]} (orchestration only; GPU work runs in the isolated venv)")


## 1 · Preflight (H100, mounts, disk, network)

In [ ]:
result_preflight = pipe.run("preflight")

## 2 · Stage and verify inputs (manifest contract, assets, source, selected audio only)

In [ ]:
result_stage_inputs = pipe.run("stage_inputs")

## 3 · Isolated pinned Python 3.11 environment + in-venv H100/decoder probe

In [ ]:
result_environment = pipe.run("environment")

## 4 · H100 VRAM-measured batch + representative runtime pilot and projection

In [ ]:
result_pilot = pipe.run("pilot")

## 5 · Full-duration frozen-encoder features for all 9504 rows (resume-safe; longest stage)

In [ ]:
result_features = pipe.run("features")

## 6 · Train six heads (seeds 41/42/43), select on val, write the immutable lock

In [ ]:
result_train = pipe.run("train")

## 7 · Locked test evaluation (masked known-label metrics)

In [ ]:
result_test_eval = pipe.run("test_eval")

## 8 · Raw-audio parity samples

In [ ]:
result_parity = pipe.run("parity")

## 9 · Single handoff ZIP (verified, published next to this notebook)

In [ ]:
result_handoff = pipe.run("handoff")

## Download link (verified copy next to this notebook)

In [ ]:
import json
from IPython.display import FileLink, Markdown, display
from dacon9504a100.handoff import expose_download

# Jupyter serves only files under its root and resolves this link against the notebook's
# directory, so the verified ZIP is exposed here (hardlink, or verified copy across
# filesystems). The durable copy stays on the object volume next to deployment.json.
_download = expose_download(DEPLOYMENT_ROOT / "DACON_Existing9504_H100_Handoff.zip.json", Path.cwd())
print(json.dumps(_download, indent=2))
display(Markdown(f"**Handoff ZIP** — sha256 `{_download['sha256']}` ({_download['bytes']:,} bytes). "
                 "Return this single file for review/submission preparation.  \n"
                 f"Durable copy: `{_download['durable_copy']}`  \n"
                 f"Download copy next to this notebook ({_download['mode']}): `{_download['path']}`"))
display(FileLink(_download["relative_path"], result_html_prefix="Download: "))
